# F3：出会いとマッチングのアルゴリズム

### マッチング成立率と長期的な関係継続の研究発表

**チーム名**：TODO　／　**メンバー**：TODO

> 教育用の架空データ・架空モデルです。実際の結婚成立率・離婚率を予測するものではなく、実在の人物の結婚適性や関係を評価するものでもありません。

## この Notebook の使い方（発表では表示しない）

- 研究発表の**テンプレート**です。上から順に表示するだけで研究の流れが伝わる構成にしてあります。
- **TODO** を、チームが実際に行った実験の結果・考察に書き換えてください（**結論や考察は書いていません**）。
- グラフは既存の `.py` から**その場で計算して**描きます。チームの実験条件に合わせてコードセルを変更してください。
- 発表のしかた：Colab / Jupyter で「すべて実行」してスクロール、または
  `jupyter nbconvert notebooks/06_research_presentation.ipynb --to slides --no-input --execute` で HTML スライドにする。
  `SAVE_FIGURES = True` で PNG を保存してスライドに貼ることもできます。

## 準備：実験環境（最初に必ず実行）

F3 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F3 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F3 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F3 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                              # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f3-matching-algorithms"     # リポジトリの中の F3 フォルダ（ルートが F3 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f3_root(p):
    return (Path(p) / "matching_algorithms.py").exists() and (Path(p) / "relationship_stability.py").exists()


def find_f3_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F3 のルート
        if is_f3_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f3_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F3 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f3_upload")
    for hit in glob.glob("/content/f3_upload/**/matching_algorithms.py", recursive=True):
        if is_f3_root(Path(hit).parent):
            return Path(hit).parent
    return None


F3_ROOT = find_f3_root()
if F3_ROOT is None:
    raise RuntimeError("F3 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F3_ROOT)
if str(F3_ROOT) not in sys.path:
    sys.path.insert(0, str(F3_ROOT))
if not (F3_ROOT / "textbook" / "fixed_queue.py").exists():
    print("⚠️ textbook/ がありません。preferences.py・matching_algorithms.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import participants as pt
import preferences as pr
import matching_algorithms as ma
import relationship_stability as rs
import evaluation as ev
import matching_charts as mc

mc.setup_style()                       # 文字を大きくするなら mc.setup_style(1.3)
groups = pt.load_participants()
scores = pr.score_table(groups)
rankings = pr.build_rankings(groups, scores)
params = rs.load_parameters()          # 関係継続モデルのパラメータ（すべて架空）
A_IDS = [p["id"] for p in groups["A"]]
B_IDS = [p["id"] for p in groups["B"]]
N_TOTAL = pt.total_participants(groups)
compat = lambda a, b: pr.pair_compatibility(a, b, scores)

SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", mc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        heads = [h for _k, h, _f in columns]
        widths = [max(len(h), *(len(d[h]) for d in data)) for h in heads]
        print("  ".join(h.rjust(w) for h, w in zip(heads, widths)))
        for d in data:
            print("  ".join(d[h].rjust(w) for h, w in zip(heads, widths)))


print("F3_ROOT:", F3_ROOT)
print(f"参加者 Group A {len(A_IDS)} 人・Group B {len(B_IDS)} 人 / 関係継続モデル（架空）:",
      {k: params[k] for k in ("baseline_hazard", "alpha", "beta", "gamma", "default_annual_stress",
                               "default_conflict_resolution", "simulation_years", "random_seed")})

## 1. 研究背景

- マッチングの良し悪しは、**成立した時点**だけで決まるのか、**その後**も見るべきか
- **TODO**：チームの動機（1〜3 行）

## 2. Research Question

**TODO**：チームが選んだ研究課題

| RQ | 内容 |
| --- | --- |
| RQ1 | マッチングアルゴリズムによって成立率は変わるか |
| RQ2 | 希望条件の厳しさは成立率と初期満足度にどう影響するか |
| RQ3 | アルゴリズムによって成立したペアの初期適合度は変わるか |
| RQ4 | 初期適合度と長期的な関係継続率には、モデル上どのような関係があるか |
| RQ5 | 成立率を最大化するアルゴリズムは、長期成立率も最大化するか |
| RQ6 | Gale–Shapley 法の数学的安定性は、長期的な関係継続を保証するか |

## 3. 仮説

**TODO**：実験の**前**に立てた仮説と理由

## 4. マッチングアルゴリズム

| アルゴリズム | しくみ | 計算量（N = 各グループの人数） |
| --- | --- | --- |
| Random | 相互承諾可能な組をランダムな順に、空いていれば成立 | O(N²)（組の列挙） |
| Greedy | A を順に処理し、空いている最も好む相手と成立（変更しない） | O(N²) 程度 |
| Gale–Shapley | 受け入れ保留方式。Blocking pair のない安定マッチング | 提案 O(N²) 回 |

**TODO**：チームで変えた点（処理順序、提案側など）

## 5. 関係継続モデル（教育用の架空モデル）

$$ p = \mathrm{clip}(\text{baseline} - \alpha \cdot \text{compatibility} - \beta \cdot \text{conflict} + \gamma \cdot \text{stress},\ 0,\ 1) $$

- 毎年、確率 $p$ で関係解消（一度解消したら戻らない）。観測期間の終わりまで続いたペアは打ち切り
- **係数はすべて架空**。初期適合度と継続の関係は**モデルの仮定**で、実証された因果関係ではない

In [ ]:
#@title 使ったパラメータ
print({k: params[k] for k in ("baseline_hazard", "alpha", "beta", "gamma", "default_annual_stress",
                              "default_conflict_resolution", "simulation_years", "random_seed")})

## 6. 実験条件

**TODO**：参加者数、希望条件、Random のシード数、反復回数、観測期間、変えたパラメータ

In [ ]:
#@title 実験の実行
MATCHING_SEEDS, REPLICATES = range(20), 300      # TODO：チームの実験条件に合わせる
res = ev.compare_algorithms(groups, rankings, scores, params, MATCHING_SEEDS, REPLICATES)
print(f"参加者 {N_TOTAL} 人、Random のシード {len(MATCHING_SEEDS)} 通り、反復 {REPLICATES} 回、観測 {params['simulation_years']} 年")

## 7. マッチング成立時点の結果

In [ ]:
#@title グラフを表示（コードは隠しています）
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
mc.plot_metric_bars(res, "matching_rate", "Matching rate", "rate", err_key="matching_rate_seed_sd", ax=axes[0], ylim=(0, 1.1))
mc.plot_metric_bars(res, "satisfaction", "Initial satisfaction", "satisfaction", fmt="{:.2f}",
                    err_key="satisfaction_seed_sd", ax=axes[1], ylim=(0, 1.1))
mc.plot_metric_bars(res, "blocking_pairs", "Blocking pairs", "count", fmt="{:.1f}", ax=axes[2])
fig.tight_layout()
show(fig, "p1_formation")

**TODO**：成立時点の結果から言えること（1 行）

## 8. 長期継続シミュレーションの結果

In [ ]:
#@title グラフを表示（コードは隠しています）
show(mc.plot_survival({r["label"]: r["survival_curve"] for r in res.values()}, "Continuation of matched pairs (model)",
                      colors={r["label"]: mc.ALG_COLORS[n] for n, r in res.items()}), "p2_survival")

**TODO**：長期継続の結果から言えること（1 行）

## 9. 統合評価

In [ ]:
#@title グラフを表示（コードは隠しています）
show(mc.plot_grouped(res, [("matching_rate", "matching rate"), ("long_term_10y", "long-term 10y")],
                     "Matching rate vs 10-year long-term matching rate (model)"), "p3_integrated")
table(list(res.values()), [("label", "algorithm", "{}"), ("matching_rate", "match rate", "{:.0%}"),
                           ("satisfaction", "satisfaction", "{:.2f}"), ("blocking_pairs", "blocking", "{:.1f}"),
                           ("continuation_10y", "10y cont.", "{:.1%}"), ("long_term_10y", "10y long-term", "{:.1%}")])

## 9. 統合評価：チームの追加実験

**TODO**：感度分析・対照条件（alpha = 0）・希望条件の変更など、チームの実験に置き換える

In [ ]:
#@title グラフを表示（コードは隠しています）
# TODO：チームの実験に置き換える（例：alpha の感度分析）
alphas = [0.0, 0.04, 0.08]
series = {n: [ev.compare_algorithms(groups, rankings, scores, rs.with_changes(params, alpha=a_), range(5), 100)[n]["continuation_10y"]
              for a_ in alphas] for n in res}
show(mc.plot_sensitivity(alphas, series, "alpha (fictional)", "10-year continuation vs alpha (model)"), "p4_team")

## 10. 考察

- **仮説との比較**：TODO
- **RQ5（成立率と長期成立率）**：TODO
- **RQ6（数学的安定性と関係継続）**：TODO
- **結果のばらつき**（シード・反復回数）：TODO

## 11. モデルの限界

- 教育用の架空シミュレーション。実際の結婚成立率・離婚率を予測しない
- 関係継続確率の係数は架空。初期適合度と継続の関係はモデルの仮定
- Blocking pair がないことは、結婚後の安定性を保証しない
- 実在の人物の結婚適性や離婚リスクを評価しない。性別・国籍・出身地などによる差別的な判断を推奨しない
- 関係解消を個人の失敗や劣等性として扱わない。関係の継続が常に望ましいとは限らない
- 現実の人間関係を大幅に単純化している

**TODO**：チームの研究で特に注意した点

## 12. 結論

**TODO**：Research Question への答え（**どの仮定の下での結論か**を必ず書く）

## 13. 今後の課題

**TODO**：Advanced（A：満足度の時間変化、B：ストレスイベント、C：確率モデルの拡張、D：再マッチング、E：多目的最適化、F：F4 との連携）でやりたいこと

## 参考文献

- D. Gale, L. S. Shapley, "College Admissions and the Stability of Marriage", *The American Mathematical Monthly*, 69(1), 1962.
- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ
- F3 リポジトリ `README.md`、データ `data/*.json`（架空）
- **TODO**：チームで使った文献

### ご清聴ありがとうございました